# 🇵🇦 Forecasting Beban Listrik Panama dengan Facebook Prophet

Notebook ini mengikuti alur: **EDA → Pengecekan Data → Persiapan Time Series → Training → Forecast → Evaluasi (MAE/RMSE/MAPE/sMAPE/R²) → Simpan Model** untuk data beban listrik nasional Panama (`nat_demand`) pada rentang 2015–2020.

> Pipeline lengkap untuk dashboard (4 model + ekspor JSON) ada di **`build_data.py`**. Notebook ini fokus pada eksplorasi dan model Prophet.


## 1. Setup / Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from prophet import Prophet

DATA_PATH = r"C:\Users\hhfzsn\Downloads\Panama_beban_listrik_per_jam_2015-2020_.xlsx"
MODEL_PATH = "D:/FreeLLM/Open Code/Project/prophet_model.pkl"
RESULTS_PATH = "D:/FreeLLM/Open Code/Project/forecast_results.pkl"

## 2. Memuat Data

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name="data")
deskripsi = pd.read_excel(DATA_PATH, sheet_name="deskripsi")

df.head()

In [ ]:
print("COLUMNS:", list(df.columns))
print("DTYPES:\n", df.dtypes)
print("SHAPE:", df.shape)

In [ ]:
deskripsi

In [ ]:
df.isna().sum()

In [ ]:
df["nat_demand"].describe()

## 3. Exploratory Data Analysis (EDA) & Pengecekan Data

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])
print("RANGE WAKTU:", df["datetime"].min(), "sampai", df["datetime"].max())
print("JUMLAH HARI UNIK:", df["datetime"].dt.date.nunique())

In [ ]:
# Agregasi harian: hanya hari lengkap 24 jam
daily = (
    df.groupby(df["datetime"].dt.date)["nat_demand"]
    .agg(["sum", "count"])
    .reset_index()
    .rename(columns={"datetime": "ds", "sum": "y"})
)

partial = daily[daily["count"] != 24]
print("Hari parsial dibuang:")
print(partial[["ds", "count"]].to_string(index=False))

daily = daily[daily["count"] == 24].drop(columns="count").reset_index(drop=True)
daily["ds"] = pd.to_datetime(daily["ds"])
print("JUMLAH HARI BERSIH:", len(daily))
daily.head()


In [ ]:
daily["y"].describe()

## 4. Training Data & Forecast (dengan Train/Test Split)

In [ ]:
# Data uji: 30 hari terakhir
train_df = daily.iloc[:-30]
test_df = daily.iloc[-30:].copy()
print("TRAIN SIZE:", len(train_df), "TEST SIZE:", len(test_df))

In [ ]:
model_eval = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95,
)
model_eval.fit(train_df[["ds", "y"]])

In [ ]:
future_eval = model_eval.make_future_dataframe(periods=30)
forecast_eval = model_eval.predict(future_eval)

merged = test_df.merge(
    forecast_eval[["ds", "yhat", "yhat_lower", "yhat_upper"]],
    on="ds",
    how="left",
)

y_true = merged["y"].values.astype(float)
y_pred = merged["yhat"].values.astype(float)

mae = float(np.mean(np.abs(y_true - y_pred)))
rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
smape = float(np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100)
ss_res = float(np.sum((y_true - y_pred) ** 2))
ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
r2 = 1 - ss_res / ss_tot

metrics = {"MAE": mae, "RMSE": rmse, "MAPE": mape, "sMAPE": smape, "R2": r2}
print("METRIK EVALUASI:")
for k, v in metrics.items():
    print(f"  {k:6s}: {v:,.4f}")


In [ ]:
fig = model_eval.plot(forecast_eval)
ax = fig.axes[0] if len(fig.axes) > 0 else plt.gca()
ax.plot(merged["ds"], merged["y"], "o", label="Beban Aktual")
ax.legend()
plt.show()

## 5. Training Model Final & Menyimpan Model

In [ ]:
model_final = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95,
)
model_final.fit(daily[["ds", "y"]])

In [ ]:
import pickle
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model_final, f)

with open(RESULTS_PATH, "wb") as f:
    pickle.dump({"test_df": merged, "metrics": metrics}, f)

print("MODEL TERSIMPAN:", MODEL_PATH)
print("HASIL FORECAST + METRIK TERSIMPAN:", RESULTS_PATH)

---
### Pipeline dashboard (4 model)

Untuk dashboard, pipeline lengkap ada di **`build_data.py`**: agregasi hari lengkap, split 30 hari test, perbandingan **Prophet, Seasonal Naive 7 hari, Holt-Winters, dan OLS harmonik**, metrik RMSE/MAE/MAPE/sMAPE/R²/AIC, serta ekspor `docs/data/forecast_results.json`.

Jalankan `python build_data.py` untuk meregenerasi model dan data dashboard.


## 6. Hasil / Kesimpulan